# PharmaSpot Morocco — run the API in Google Colab

This notebook **installs the backend**, starts **FastAPI**, and gives you a **public HTTPS link** (via ngrok) so you can point **Vercel** at it with `VITE_API_BASE_URL`.

**Limits (be honest):**
- Colab **disconnects** when idle or after a time limit; the link **stops working** until you re-run cells.
- Free ngrok URLs **change each session** unless you add an [ngrok auth token](https://dashboard.ngrok.com/get-started/your-authtoken).
- First analysis can be **slow** (Overpass + WorldPop); heavy cities may **run out of RAM** on free Colab.
- The **React map UI** is not in this notebook — deploy the `frontend/` on **Vercel** and set `VITE_API_BASE_URL` to the ngrok URL below.


## 1) Clone the project

Change `REPO` if you use a fork.

In [ ]:
import shutil
import subprocess

REPO = "https://github.com/adilchetouani-rgb/pharmacie.git"
BRANCH = "main"
DEST = "/content/pharmacie"

# Colab sometimes fails shallow clone (exit 128); full clone is more reliable.
subprocess.run(["apt-get", "update", "-qq"], check=False)
subprocess.run(["apt-get", "install", "-y", "-qq", "git"], check=False)

shutil.rmtree(DEST, ignore_errors=True)

def _clone(cmd):
    p = subprocess.run(cmd, capture_output=True, text=True)
    return p.returncode, p.stdout, p.stderr

code, out, err = _clone(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, DEST]
)
if code != 0:
    print("Shallow clone stderr:", err or out)
    shutil.rmtree(DEST, ignore_errors=True)
    code, out, err = _clone(["git", "clone", REPO, DEST])
    if code != 0:
        raise RuntimeError(f"git clone failed:\n{err}\n{out}")
    subprocess.run(["git", "-C", DEST, "checkout", BRANCH], check=True)

print("Clone OK.")
%cd /content/pharmacie

## 2) System libraries (GDAL / spatialindex) + Python packages

This can take **several minutes**.

In [ ]:
%%capture cap --no-stderr
!apt-get update -qq
!apt-get install -y -qq gdal-bin libgdal-dev libspatialindex-dev g++
!pip install -q pyngrok
!pip install -q -r /content/pharmacie/backend/requirements.txt

## 3) ngrok authtoken (required)

Since 2024, **anonymous tunnels are blocked**. You must:

1. Sign up at [ngrok](https://dashboard.ngrok.com/signup) and **verify your email**.
2. Open [Your authtoken](https://dashboard.ngrok.com/get-started/your-authtoken) and copy the token.
3. In Colab: **Secrets** (key icon) → **Add secret** → **Name:** exactly `NGROK_AUTHTOKEN` → **Value:** paste token → enable **Notebook access** for this notebook.
4. Run the next cell — it should print `OK: NGROK_AUTHTOKEN is set`.

In [ ]:
import os


def _read_ngrok_token():
    """Colab Secrets: name must be exactly NGROK_AUTHTOKEN (no extra spaces)."""
    try:
        from google.colab import userdata

        return userdata.get("NGROK_AUTHTOKEN").strip()
    except Exception:
        pass
    return os.environ.get("NGROK_AUTHTOKEN", "").strip()


tok = _read_ngrok_token()
if tok:
    print("OK: NGROK_AUTHTOKEN is set (length %d)." % len(tok))
else:
    print(
        "MISSING: add Colab Secret NGROK_AUTHTOKEN (key icon) or set env NGROK_AUTHTOKEN.\n"
        "On ngrok.com: verify your email, then copy authtoken from the dashboard."
    )

## 4) Start the API (background) + public URL

Run this once. If you change code, **restart runtime** and re-run from the top.

In [ ]:
import os
import subprocess
import time
import sys


def _read_ngrok_token():
    try:
        from google.colab import userdata

        return userdata.get("NGROK_AUTHTOKEN").strip()
    except Exception:
        return os.environ.get("NGROK_AUTHTOKEN", "").strip()


TOKEN = _read_ngrok_token()
if not TOKEN:
    raise RuntimeError(
        "ngrok needs an authtoken (ERR_NGROK_4018).\n"
        "1) Create/verify account: https://dashboard.ngrok.com/signup\n"
        "2) Copy authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\n"
        "3) Colab → Secrets (key) → name NGROK_AUTHTOKEN → paste token → allow notebook access\n"
        "4) Re-run this cell."
    )

from pyngrok import ngrok

ngrok.set_auth_token(TOKEN)
ngrok.kill()

# Stop previous uvicorn if re-running
!pkill -f "uvicorn main:app" 2>/dev/null || true
time.sleep(1)

os.chdir("/content/pharmacie/backend")
sys.path.insert(0, "/content/pharmacie/backend")

log = open("/content/uvicorn.log", "w")
proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=log,
    stderr=subprocess.STDOUT,
    cwd="/content/pharmacie/backend",
)
time.sleep(3)

tunnel = ngrok.connect(8000)
BASE = getattr(tunnel, "public_url", str(tunnel)).rstrip("/")
print("\n=== Backend base URL (use for Vercel VITE_API_BASE_URL) ===")
print(BASE)
print("\nTest:", BASE + "/api/health")
print("\nKeep this tab open; Colab stops when disconnected.")

## 5) Quick test (optional)

In [ ]:
import urllib.request
import json

with urllib.request.urlopen(BASE + "/api/health", timeout=30) as r:
    print(json.loads(r.read().decode()))

## Vercel

Set **`VITE_API_BASE_URL`** to the printed **`BASE`** (same as ngrok HTTPS URL), then redeploy the frontend.

---

**If install fails:** open the captured pip output by changing the install cell to remove `%%capture`, re-run, and read errors. Some Colab runtimes need a **Runtime → Restart session** after apt installs.